In [28]:
import torchvision
import torchvision.transforms as transforms

# download MNIST dataset
train_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

test_dataset = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transforms.ToTensor()
)
print(len(train_dataset))
print(len(test_dataset))

60000
10000


In [29]:
# Each data contains an image and its label
image, label = train_dataset[0]
print(image.shape)
print(label)


torch.Size([1, 28, 28])
5


In [30]:
# Get train data and test data
import torch

images = []
labels = []
def getdata(dataset):
    for i in range(0, len(dataset)):
        image, label = dataset[i]
        images.append(image)
        labels.append(label)
    
    # torch.stack 的作用是沿着一个新的维度把这些张量拼起来。默认是在 dim=0 位置
    x = torch.stack(images)
    # print(x.shape) # torch.Size([60000, 1, 28, 28])

    # MLP needs the input to be a vector, so flatten each image to a vector
    x = x.view(x.size(0), -1) # reshape, x.size(0) is 60000, -1 表示沿这个维度自动计算
    # print(x.shape) #torch.Size([60000, 784])
    
    # translate a list to a tensor
    y = torch.tensor(labels)
    # print(y.shape) # torch.Size([60000])

    return x, y

xtrain = []
ytrain = []
xtrain, ytrain = getdata(train_dataset)

    

In [31]:
# can also use Dataloader to do this
from torch.utils.data import DataLoader

# batch_size:每次返回64张照片，训练的时候需要打乱数据顺序
train_loader = DataLoader (
    train_dataset,
    batch_size = 64,
    shuffle = True
)

test_loader = DataLoader (
    test_dataset,
    batch_size = 64,
    shuffle = False
)

In [32]:
# Use MLP to train the network
import torch.nn as nn
import torch.optim as optim

input_size = 1 * 28 * 28
# 增加隐藏层可以表达更丰富的信息，但是主要是为了引入非线性激活函数，否则一直是在做线性变换
hidden_size = 32
output_size = 10

# Use pytorch Sequential
model = nn.Sequential(
    nn.Linear(input_size, hidden_size),
    nn.ReLU(),
    nn.Linear(hidden_size, output_size)
)

# 这一步把模型中所有可训练参数传给 optimizer，每一层的weights和bias会被model.parameters()收集
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    for images, labels in train_loader:
        images = images.view(images.size(0), -1) # torch.Size([64, 1 * 28 * 28])
        outputs = model(images)
        
        loss = criterion(outputs, labels)
        # 把每个batch梯度清零，因为 PyTorch 默认会累加梯度，如果不清零，下一次 backward 会叠加上一次的梯度
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 0.5063151121139526
Epoch 1, Loss: 0.5016196370124817
Epoch 2, Loss: 0.47429153323173523
Epoch 3, Loss: 0.6450448632240295
Epoch 4, Loss: 0.16820277273654938
